# Data Engineering & Preprocessing

This notebook is the preprocessing pipeline for our flight cancellation prediction project. It takes the raw `flights_with_weather.csv` (~3.23M flights, 44 columns, 2019-2023), cleans it up, engineers the features we need, and saves everything as ready-to-use parquet files in `artifacts/`.

The idea is that everyone on the team runs their models on the same preprocessed data, so results are comparable. Just run all cells in order — the last cell exports everything. Check `artifacts/README.md` for details on what each file contains.

**What's in here:**
1. Load the dataset from Kaggle
2. Check data quality and figure out which columns to keep vs drop
3. Engineer features (cyclical time encoding, OOF target encoding, imputation, scaling)
4. Export all artifacts

In [23]:
!pip install -q kagglehub category_encoders

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, gc

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import category_encoders as ce

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 50)


## Load the data

We're pulling the dataset straight from Kaggle. It already has weather data merged in — 32 flight columns plus 12 weather columns (6 for origin airport, 6 for destination).

In [25]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'huskydawg/flight-cancellation2019-2023-full-with-weather',
    'flights_with_weather.csv',
)

df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
df['CANCELLED'] = df['CANCELLED'].fillna(0).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Date range:    {df["FL_DATE"].min().date()} to {df["FL_DATE"].max().date()}')
print(f'Memory usage:  {df.memory_usage(deep=True).sum() / 1e6:.0f} MB')
display(df.head())

C:\Users\raphz\AppData\Local\Temp\ipykernel_30156\2675567350.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Dataset shape: (3234981, 44)
Date range:    2019-01-01 to 2023-12-31
Memory usage:  2397 MB


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,origin_temp_mean_c,origin_wind_speed_max_kmh,origin_pressure_msl_hpa,origin_humidity_pct,origin_precipitation_mm,origin_snowfall_cm,dest_temp_mean_c,dest_wind_speed_max_kmh,dest_pressure_msl_hpa,dest_humidity_pct,dest_precipitation_mm,dest_snowfall_cm
0,2019-01-01,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1404,LAX,"Los Angeles, CA",SLC,"Salt Lake City, UT",601,600.0,-1.0,16.0,616.0,848.0,4.0,900,852.0,-8.0,0,NaN,0.0,119.0,112.0,92.0,590.0,NaN,NaN,NaN,NaN,NaN,9.4,26.3,1019.2,33.0,0.0,0.00,-9.7,16.4,1033.0,56.0,0.0,0.00
1,2019-01-01,SkyWest Airlines Inc.,SkyWest Airlines Inc.: OO,OO,20304,5016,LWB,"Lewisburg, WV",ORD,"Chicago, IL",1745,1846.0,61.0,11.0,1857.0,1929.0,43.0,1844,2012.0,88.0,0,NaN,0.0,119.0,146.0,92.0,489.0,0.0,0.0,27.0,0.0,61.0,9.0,20.7,1018.1,85.0,1.8,0.00,-1.9,14.2,1024.7,83.0,0.3,0.14
2,2019-01-01,Mesa Airlines Inc.,Mesa Airlines Inc.: YV,YV,20378,5733,LFT,"Lafayette, LA",DFW,"Dallas/Fort Worth, TX",626,623.0,-3.0,19.0,642.0,745.0,11.0,800,756.0,-4.0,0,NaN,0.0,94.0,93.0,63.0,351.0,NaN,NaN,NaN,NaN,NaN,15.3,17.2,1019.7,80.0,0.8,0.00,3.4,25.5,1025.5,71.0,0.0,0.00
3,2019-01-01,Republic Airline,Republic Airline: YX,YX,20452,4660,TLH,"Tallahassee, FL",CLT,"Charlotte, NC",2023,2013.0,-10.0,13.0,2026.0,2123.0,18.0,2149,2141.0,-8.0,0,NaN,0.0,86.0,88.0,57.0,386.0,NaN,NaN,NaN,NaN,NaN,21.2,13.0,1020.5,91.0,0.0,0.00,16.7,22.5,1018.0,92.0,1.0,0.00
4,2019-01-01,SkyWest Airlines Inc.,SkyWest Airlines Inc.: OO,OO,20304,5426,DEN,"Denver, CO",ASE,"Aspen, CO",755,804.0,9.0,33.0,837.0,905.0,5.0,855,910.0,15.0,0,NaN,0.0,60.0,66.0,28.0,125.0,9.0,0.0,6.0,0.0,0.0,-12.0,11.0,1032.0,53.0,0.2,0.14,-11.0,7.8,1023.0,71.0,1.1,0.84


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3234981 entries, 0 to 3234980
Data columns (total 44 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   FL_DATE                    datetime64[ns]
 1   AIRLINE                    object        
 2   AIRLINE_DOT                object        
 3   AIRLINE_CODE               object        
 4   DOT_CODE                   int64         
 5   FL_NUMBER                  int64         
 6   ORIGIN                     object        
 7   ORIGIN_CITY                object        
 8   DEST                       object        
 9   DEST_CITY                  object        
 10  CRS_DEP_TIME               int64         
 11  DEP_TIME                   float64       
 12  DEP_DELAY                  float64       
 13  TAXI_OUT                   float64       
 14  WHEELS_OFF                 float64       
 15  WHEELS_ON                  float64       
 16  TAXI_IN                    float64  

---
## Quality check

Before we start preprocessing, we need to understand what we're working with — how much data is missing, and which columns might leak information about the target.

In [27]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
}).sort_values('Missing %', ascending=False)

print('Missing values:')
display(missing[missing['Missing Count'] > 0])

# Compare missingness for cancelled vs operated flights
# If a column is always NaN when cancelled, it's post-departure info (leakage)
cancelled = df[df['CANCELLED'] == 1]
operated = df[df['CANCELLED'] == 0]

leak_check = pd.DataFrame({
    'NaN % (Cancelled)': (cancelled.isnull().mean() * 100).round(1),
    'NaN % (Operated)': (operated.isnull().mean() * 100).round(1),
})

print('\nColumns that are basically always missing for cancelled flights (leakage):')
display(leak_check[leak_check['NaN % (Cancelled)'] > 90].sort_values(
    'NaN % (Cancelled)', ascending=False))

del cancelled, operated
gc.collect()

Missing values:


,Missing Count,Missing %
CANCELLATION_CODE,3149230,97.35
DELAY_DUE_CARRIER,2665058,82.38
DELAY_DUE_NAS,2665058,82.38
DELAY_DUE_SECURITY,2665058,82.38
DELAY_DUE_LATE_AIRCRAFT,2665058,82.38
DELAY_DUE_WEATHER,2665058,82.38
AIR_TIME,93203,2.88
ELAPSED_TIME,93203,2.88
ARR_DELAY,93203,2.88
ARR_TIME,86598,2.68



Columns that are basically always missing for cancelled flights (leakage):


,NaN % (Cancelled),NaN % (Operated)
WHEELS_ON,100.0,0.0
TAXI_IN,100.0,0.0
ARR_TIME,100.0,0.0
ARR_DELAY,100.0,0.2
ELAPSED_TIME,100.0,0.2
AIR_TIME,100.0,0.2
DELAY_DUE_CARRIER,100.0,81.9
DELAY_DUE_WEATHER,100.0,81.9
DELAY_DUE_NAS,100.0,81.9
DELAY_DUE_SECURITY,100.0,81.9


0

### How we're classifying the columns

Based on the EDA and the leakage check, here's what we're keeping and what we're dropping:

- **Target:** `CANCELLED` — this is what we're predicting
- **Pre-departure features (keep):** `FL_DATE`, `AIRLINE_CODE`, `ORIGIN`, `DEST`, `CRS_DEP_TIME`, `CRS_ELAPSED_TIME`, `DISTANCE` — all known before the flight departs
- **Weather features (keep):** 12 columns (6 origin + 6 destination) — temperature, wind, pressure, humidity, precipitation, snowfall
- **Post-departure columns (drop):** `DEP_TIME`, `DEP_DELAY`, `TAXI_OUT`, actual times, delay breakdowns, `CANCELLATION_CODE`, `DIVERTED` — these only exist after a flight operates or gets cancelled, so using them would be cheating
- **Redundant identifiers (drop):** `AIRLINE`, `AIRLINE_DOT`, `DOT_CODE`, `FL_NUMBER`, `ORIGIN_CITY`, `DEST_CITY`, `CRS_ARR_TIME` — duplicates or not useful for prediction
- **COVID flag (engineered):** We'll create an `IS_COVID` binary (Mar 2020 – Jun 2021) and export two versions of the dataset — with and without it — so we can test whether it helps


## Feature preprocessing

We need to split the data *before* fitting any encoders or scalers, otherwise we'd be leaking validation/test info into the training set.

Drop bad columns → engineer time features → temporal split → target-encode categoricals (OOF on train) → impute missing values → scale everything.

In [28]:
# Drop all the post-departure leakage columns and redundant identifiers
y = df['CANCELLED'].values

LEAK_COLS = [
    'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON',
    'TAXI_IN', 'ARR_TIME', 'ARR_DELAY', 'ELAPSED_TIME', 'AIR_TIME',
    'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
    'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT',
    'CANCELLATION_CODE', 'DIVERTED',
]

REDUNDANT_COLS = [
    'AIRLINE', 'AIRLINE_DOT', 'DOT_CODE', 'FL_NUMBER',
    'ORIGIN_CITY', 'DEST_CITY', 'CRS_ARR_TIME',
]

df = df.drop(columns=LEAK_COLS + REDUNDANT_COLS + ['CANCELLED'])

print(f'After dropping leakage/redundant + target: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

After dropping leakage/redundant + target: (3234981, 19)
Remaining columns: ['FL_DATE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'origin_temp_mean_c', 'origin_wind_speed_max_kmh', 'origin_pressure_msl_hpa', 'origin_humidity_pct', 'origin_precipitation_mm', 'origin_snowfall_cm', 'dest_temp_mean_c', 'dest_wind_speed_max_kmh', 'dest_pressure_msl_hpa', 'dest_humidity_pct', 'dest_precipitation_mm', 'dest_snowfall_cm']


In [29]:
# Extract time features and encode them cyclically (sin/cos)
# so the model knows that December and January are neighbors, not 11 apart
df['MONTH']       = df['FL_DATE'].dt.month
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek
df['DEP_HOUR']    = (df['CRS_DEP_TIME'].fillna(0) // 100).clip(0, 23).astype(int)

df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
df['DOW_SIN']   = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['DOW_COS']   = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['HOUR_SIN']  = np.sin(2 * np.pi * df['DEP_HOUR'] / 24)
df['HOUR_COS']  = np.cos(2 * np.pi * df['DEP_HOUR'] / 24)
df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 5).astype(int)
df['IS_COVID']   = ((df['FL_DATE'] >= '2020-03-01') & (df['FL_DATE'] <= '2021-06-30')).astype(int)

df = df.drop(columns=['CRS_DEP_TIME', 'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR'])

print(f'After feature engineering: {df.shape}')
print(f'IS_COVID distribution: {df["IS_COVID"].value_counts().to_dict()}')

After feature engineering: (3234981, 26)
IS_COVID distribution: {0: 2608922, 1: 626059}


### Temporal split

We split by date. Train on everything up to mid-2022, validate on the second half of 2022, and test on all of 2023.

In [30]:
train_mask = (df['FL_DATE'] < '2022-07-01').values
val_mask   = ((df['FL_DATE'] >= '2022-07-01') & (df['FL_DATE'] < '2023-01-01')).values
test_mask  = (df['FL_DATE'] >= '2023-01-01').values

X_train = df[train_mask].drop(columns=['FL_DATE']).reset_index(drop=True)
X_val   = df[val_mask].drop(columns=['FL_DATE']).reset_index(drop=True)
X_test  = df[test_mask].drop(columns=['FL_DATE']).reset_index(drop=True)

y_train = y[train_mask]
y_val   = y[val_mask]
y_test  = y[test_mask]

del df
gc.collect()

for name, X_s, y_s in [('Train', X_train, y_train),
                        ('Val',   X_val,   y_val),
                        ('Test',  X_test,  y_test)]:
    rate = y_s.mean() * 100
    print(f'{name:5s}: {len(X_s):>10,} rows  |  cancellation rate: {rate:.2f}%')

Train:  2,186,803 rows  |  cancellation rate: 2.92%
Val  :    349,591 rows  |  cancellation rate: 2.16%
Test :    698,587 rows  |  cancellation rate: 2.06%


In [31]:
# OOF target encoding for the high-cardinality categoricals.
# Each training row gets encoded using only data from the OTHER folds,
# so no row sees its own target. Then we fit one final encoder on all
# of train for transforming val/test.
from sklearn.model_selection import StratifiedKFold

cat_cols = ['ORIGIN', 'DEST', 'AIRLINE_CODE']
K = 5

X_train_encoded = X_train.copy()
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

for fold_idx, (fit_idx, oof_idx) in enumerate(skf.split(X_train, y_train)):
    fold_encoder = ce.TargetEncoder(cols=cat_cols, smoothing=1.0)
    fold_encoder.fit(X_train.iloc[fit_idx][cat_cols], y_train[fit_idx])
    X_train_encoded.iloc[oof_idx, X_train_encoded.columns.get_indexer(cat_cols)] = (
        fold_encoder.transform(X_train.iloc[oof_idx][cat_cols]).values
    )

X_train[cat_cols] = X_train_encoded[cat_cols]

encoder = ce.TargetEncoder(cols=cat_cols, smoothing=1.0)
encoder.fit(X_train[cat_cols], y_train)

X_val[cat_cols]  = encoder.transform(X_val[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

del X_train_encoded

print(f'OOF target-encoded (K={K}): {cat_cols}')
print(f'\nEncoded ORIGIN values (train sample):')
display(X_train['ORIGIN'].describe().round(4))

OOF target-encoded (K=5): ['ORIGIN', 'DEST', 'AIRLINE_CODE']

Encoded ORIGIN values (train sample):


count     2.186803e+06
unique    1.660000e+03
top       1.760000e-02
freq      2.302800e+04
Name: ORIGIN, dtype: float64

In [32]:
# Fill in missing values for the numeric columns using median
impute_cols = ['CRS_ELAPSED_TIME', 'DISTANCE']

imputer = SimpleImputer(strategy='median')
X_train[impute_cols] = imputer.fit_transform(X_train[impute_cols])
X_val[impute_cols]   = imputer.transform(X_val[impute_cols])
X_test[impute_cols]  = imputer.transform(X_test[impute_cols])

# Make sure there are no stray non-numeric columns left
non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f'Coercing non-numeric columns to float: {non_numeric}')
    for col in non_numeric:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_val[col]   = pd.to_numeric(X_val[col], errors='coerce')
        X_test[col]  = pd.to_numeric(X_test[col], errors='coerce')

# Scale everything to mean=0, std=1
feature_names = X_train.columns.tolist()
scaler = StandardScaler()

X_train_all = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_names)
X_val_all = pd.DataFrame(
    scaler.transform(X_val), columns=feature_names)
X_test_all = pd.DataFrame(
    scaler.transform(X_test), columns=feature_names)

# Create two versions: one with IS_COVID, one without (for the ablation study)
X_train_covid = X_train_all.copy()
X_val_covid   = X_val_all.copy()
X_test_covid  = X_test_all.copy()

X_train_no_covid = X_train_all.drop(columns=['IS_COVID'])
X_val_no_covid   = X_val_all.drop(columns=['IS_COVID'])
X_test_no_covid  = X_test_all.drop(columns=['IS_COVID'])

print(f'With IS_COVID:    {X_train_covid.shape[1]} features')
print(f'Without IS_COVID: {X_train_no_covid.shape[1]} features')
print(f'Features (with): {X_train_covid.columns.tolist()}')
print(f'\nScaled training set (mean ~ 0, std ~ 1):')
display(X_train_covid.describe().round(3))

Coercing non-numeric columns to float: ['AIRLINE_CODE', 'ORIGIN', 'DEST']
With IS_COVID:    25 features
Without IS_COVID: 24 features
Features (with): ['AIRLINE_CODE', 'ORIGIN', 'DEST', 'CRS_ELAPSED_TIME', 'DISTANCE', 'origin_temp_mean_c', 'origin_wind_speed_max_kmh', 'origin_pressure_msl_hpa', 'origin_humidity_pct', 'origin_precipitation_mm', 'origin_snowfall_cm', 'dest_temp_mean_c', 'dest_wind_speed_max_kmh', 'dest_pressure_msl_hpa', 'dest_humidity_pct', 'dest_precipitation_mm', 'dest_snowfall_cm', 'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS', 'HOUR_SIN', 'HOUR_COS', 'IS_WEEKEND', 'IS_COVID']

Scaled training set (mean ~ 0, std ~ 1):


,AIRLINE_CODE,ORIGIN,DEST,CRS_ELAPSED_TIME,DISTANCE,origin_temp_mean_c,origin_wind_speed_max_kmh,origin_pressure_msl_hpa,origin_humidity_pct,origin_precipitation_mm,origin_snowfall_cm,dest_temp_mean_c,dest_wind_speed_max_kmh,dest_pressure_msl_hpa,dest_humidity_pct,dest_precipitation_mm,dest_snowfall_cm,MONTH_SIN,MONTH_COS,DOW_SIN,DOW_COS,HOUR_SIN,HOUR_COS,IS_WEEKEND,IS_COVID
count,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000,2186803.000
mean,-0.000,-0.000,-0.000,-0.000,-0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,0.000,0.000,-0.000,-0.000,-0.000,0.000,-0.000,-0.000,0.000,-0.000,0.000,0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-1.952,-3.315,-3.461,-1.739,-1.328,-5.554,-2.348,-8.229,-3.608,-0.406,-0.133,-5.969,-2.349,-7.423,-3.601,-0.407,-0.133,-1.541,-1.397,-1.410,-1.252,-1.227,-1.161,-0.614,-0.633
25%,-0.574,-0.520,-0.536,-0.726,-0.738,-0.705,-0.738,-0.621,-0.483,-0.406,-0.133,-0.706,-0.738,-0.620,-0.480,-0.407,-0.133,-0.832,-1.207,-1.132,-1.252,-1.049,-0.909,-0.614,-0.633
50%,-0.051,0.025,0.049,-0.233,-0.263,0.118,-0.154,-0.058,0.187,-0.406,-0.133,0.117,-0.153,-0.058,0.188,-0.407,-0.133,-0.123,0.024,-0.010,-0.307,-0.241,-0.220,-0.614,-0.633
75%,0.637,0.650,0.613,0.414,0.407,0.809,0.587,0.582,0.745,-0.098,-0.133,0.819,0.589,0.582,0.746,-0.112,-0.133,1.105,0.734,1.113,0.872,1.044,0.720,1.629,1.579
max,3.145,17.793,24.588,7.928,8.587,2.567,12.428,5.128,1.805,36.928,50.840,2.566,12.441,6.605,1.804,24.709,43.559,1.296,1.445,1.390,1.397,1.434,2.600,1.629,1.579


---
## Export train/val/test sets

The dataset is heavily imbalanced (~2.6% cancelled). Our group should handle this in their own models(e.g. `class_weight='balanced'`, `scale_pos_weight`, etc.) since the right approach depends on the model.

We export two versions of the features (with and without `IS_COVID`).

In [33]:
OUT_DIR = 'artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

print(f'Class distribution (train):')
print(f'  Operated:  {n_neg:>10,}  ({n_neg/len(y_train)*100:.2f}%)')
print(f'  Cancelled: {n_pos:>10,}  ({n_pos/len(y_train)*100:.2f}%)')
print(f'  Ratio:     {n_neg/n_pos:.1f} : 1')

X_train_covid.to_parquet(f'{OUT_DIR}/X_train_with_covid.parquet')
X_val_covid.to_parquet(f'{OUT_DIR}/X_val_with_covid.parquet')
X_test_covid.to_parquet(f'{OUT_DIR}/X_test_with_covid.parquet')

X_train_no_covid.to_parquet(f'{OUT_DIR}/X_train_no_covid.parquet')
X_val_no_covid.to_parquet(f'{OUT_DIR}/X_val_no_covid.parquet')
X_test_no_covid.to_parquet(f'{OUT_DIR}/X_test_no_covid.parquet')

pd.DataFrame({'CANCELLED': y_train}).to_parquet(f'{OUT_DIR}/y_train.parquet')
pd.DataFrame({'CANCELLED': y_val}).to_parquet(f'{OUT_DIR}/y_val.parquet')
pd.DataFrame({'CANCELLED': y_test}).to_parquet(f'{OUT_DIR}/y_test.parquet')

print(f'\nExported to {OUT_DIR}/:')
print(f'  With IS_COVID:    {X_train_covid.shape[1]} features')
print(f'  Without IS_COVID: {X_train_no_covid.shape[1]} features')
print()
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f'  {f}: {sz:.1f} MB')


Class distribution (train):
  Operated:   2,123,017  (97.08%)
  Cancelled:     63,786  (2.92%)
  Ratio:     33.3 : 1



Exported to artifacts/:
  With IS_COVID:    25 features
  Without IS_COVID: 24 features

  X_test_no_covid.parquet: 10.6 MB
  X_test_with_covid.parquet: 10.6 MB
  X_train_no_covid.parquet: 41.7 MB
  X_train_with_covid.parquet: 41.7 MB
  X_val_no_covid.parquet: 5.1 MB
  X_val_with_covid.parquet: 5.1 MB
  y_test.parquet: 0.0 MB
  y_train.parquet: 0.1 MB
  y_val.parquet: 0.0 MB
